# PDF Explainer AI - Backend Pipeline Demonstration

This notebook demonstrates the core backend architecture of our application, completely separated from the Streamlit UI. It loads unstructured data from our `Dataset` folder, preprocesses it, embeds the text using Pinecone, and finally processes a RAG query through our Vector Database and LLM generator.

In [ ]:
import os
import re
import tiktoken
import requests
import PyPDF2
import streamlit as st
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
import warnings
warnings.filterwarnings('ignore')

# Automatically load API keys from the Streamlit secrets file (.streamlit/secrets.toml)
GROQ_API_KEY = st.secrets.get("GROQ_API_KEY", "")
PINECONE_API_KEY = st.secrets.get("PINECONE_API_KEY", "")

if not GROQ_API_KEY or not PINECONE_API_KEY:
    print("Warning: API Keys not found in .streamlit/secrets.toml")
else:
    print("Libraries imported and API keys loaded successfully!")

## 1. Data Ingestion & Preprocessing
We load a testing document from our Dataset folder, extract its contents using `PyPDF2`, and preprocess the extracted string.

In [ ]:
# Point this to one of the PDFs in your new Dataset/Raw_PDFs folder
pdf_path = "Dataset/Raw_PDFs/sample_document.pdf"

text = ""
try:
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text
except FileNotFoundError:
    print(f"Please put a PDF named 'sample_document.pdf' inside '{pdf_path}'")
    text = "This is dummy text used as a fallback if the PDF isn't placed in the folder yet."

# Preprocess step (matches functions.py)
text = re.sub(r'\n+', ' ', text)
text = re.sub(r'\s+', ' ', text).strip()

print(f"Extracted and preprocessed {len(text)} characters from the dataset.")
print("Preview:", text[:200], "...")

## 2. Text Chunking
We chunk the data into manageable contextual sizes using OpenAI's `tiktoken`.

In [ ]:
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(text)

chunks = []
max_tokens = 500
overlap = 50
start = 0

while start < len(tokens):
    end = start + max_tokens
    chunk_tokens = tokens[start:end]
    chunk_text = encoding.decode(chunk_tokens)
    if chunk_text.strip():
        chunks.append(chunk_text.strip())
    start = end - overlap

print(f"Divided the document into {len(chunks)} contextual chunks for AI processing.")

## 3. Summarization Generation
We query our Groq endpoint leveraging Llama-3.1 to summarize the chunked information.

In [ ]:
def test_summary(chunk):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [{"role": "user", "content": f"Summarize this text in 50 words:\n{chunk}"}],
        "max_tokens": 150
    }
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"].strip()
        return f"Error API. Status Code: {response.status_code}"
    except Exception as e:
        return str(e)

print("--- AI EVALUATION OUTPUT ---")
# Only loop through the first 2 chunks so the notebook runs quickly
for i, chunk in enumerate(chunks[:2]):
    print(f"\nChunk {i+1} Summary:")
    print(test_summary(chunk))


## 4. RAG Chat Pipeline (Pinecone Vector Retrieval + LLM Generation)
We embed the chunks and store them in Pinecone. Then, given a user question, we retrieve the most relevant context and pass both the question and the context to the LLM to generate an informed answer.

In [ ]:
print("Loading Embedding Model...")
try:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    
    # Initialize Pinecone
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index = pc.Index("pdfexplainer078") # Using your existing index
    
    # 1. Embed and Store the first chunk
    vector = model.encode(chunks[0]).tolist()
    print("Upserting chunk to Pinecone vector database...")
    index.upsert(
        vectors=[{"id": "notebook-test-chunk-0", "values": vector, "metadata": {"text": chunks[0]}}], 
        namespace="notebook_test"
    )
    
    # 2. Simulate User Query Retrieval
    question = "What is the main topic of this text?"
    q_vector = model.encode(question).tolist()
    results = index.query(vector=q_vector, top_k=1, include_metadata=True, namespace="notebook_test")
    
    print("\n--- PINEPONE RETRIEVAL ---")
    print("User Question:", question)
    
    if results['matches']:
        context = results['matches'][0]['metadata']['text']
        print("Most relevant context retrieved:\n", context[:200], "...\n")
        
        # 3. Request LLM Answer with Context (RAG completion)
        print("--- FINAL LLM RAG ANSWER ---")
        print("Sending Context + Query to LLM for final response...")
        
        url = "https://api.groq.com/openai/v1/chat/completions"
        headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
        prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer the question using only the context provided."
        payload = {
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 200
        }
        
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            final_answer = response.json()["choices"][0]["message"]["content"].strip()
            print("\n\u2728 Final AI Answer:")
            print("--------------------------------------------------")
            print(final_answer)
            print("--------------------------------------------------")
        else:
            print(f"API Error during RAG generation. Status Code: {response.status_code}")
    else:
        print("No matches found. Check vector embedding dimensionality.")
except Exception as e:
    print(f"Error connecting to Pinecone or embedding model: {e}\nMake sure you provided valid API keys!")